In [ ]:
"""
State space modeling functions
"""

import pandas as pd
import numpy as np
from config import settings


def assign_initial_state_xy(row: pd.Series) -> tuple:
    """Assign initial state based on lab values."""
    bleeding_score = 0
    thrombus_score = 0
    
    # Bleeding score calculations
    if pd.notna(row["APTT"]):
        if row["APTT"] > 60:
            bleeding_score += 2
        elif row["APTT"] > 45:
            bleeding_score += 1
    
    if pd.notna(row["PT"]):
        if row["PT"] > 25:
            bleeding_score += 2
        elif row["PT"] > 18:
            bleeding_score += 1
    
    if pd.notna(row["TT"]) and row["TT"] > 20:
        bleeding_score += 1
    
    if pd.notna(row["PLT"]):
        if row["PLT"] < 50:
            bleeding_score += 2
        elif row["PLT"] < 100:
            bleeding_score += 1
    
    if pd.notna(row["Fib"]) and row["Fib"] < 2.0:
        bleeding_score += 1
    
    # Thrombosis score calculations
    if pd.notna(row["TAT"]):
        if row["TAT"] > 20:
            thrombus_score += 2
        elif row["TAT"] > 10:
            thrombus_score += 1
    
    if pd.notna(row["PIC"]):
        if row["PIC"] > 3:
            thrombus_score += 2
        elif row["PIC"] > 1.5:
            thrombus_score += 1
    
    if pd.notna(row["PAIC"]) and row["PAIC"] > 17.1:
        thrombus_score += 1
    
    if pd.notna(row["TM"]) and row["TM"] > 5.0:
        thrombus_score += 1
    
    if pd.notna(row["DD"]):
        if row["DD"] > 5:
            thrombus_score += 2
        elif row["DD"] > 1:
            thrombus_score += 1
    
    if pd.notna(row["Fib"]) and row["Fib"] > 4.0:
        thrombus_score += 1
    
    return bleeding_score, thrombus_score


def build_dynamic_states(
    df: pd.DataFrame,
    id_col: str = "ID",
    id2_col: str = "ID2",
    time_col: str = "Time",
    value_cols: list = None,
    directions: dict = None,
    max_step_delta: float = 1.0,
    total_delta_limit: float = 3.0
) -> pd.DataFrame:
    """Build dynamic state space representation."""
    
    if value_cols is None:
        value_cols = settings.KEY_COLS
    if directions is None:
        directions = {
            'TAT': -1, 'PIC': -1, 'PAIC': -1, 'TM': -1,
            'PT': -1, 'APTT': -1, 'TT': -1, 'Fib': 1,
            'DD': -1, 'PLT': 1, 'Hb': 1, 'HCT': 1
        }
    
    df = df.sort_values([id_col, id2_col, time_col]).copy()
    df["Y_x"] = np.nan
    df["Y_y"] = np.nan
    
    for (pid, pid2), group in df.groupby([id_col, id2_col]):
        prev_vals = None
        prev_state = None
        cumulative_dx, cumulative_dy = 0.0, 0.0
        
        for idx, row in group.iterrows():
            is_treat = pd.notna(row.get("Treat")) and len(str(row.get("Treat"))) > 0
            
            if prev_state is None:
                Y_x, Y_y = assign_initial_state_xy(row)
                prev_state = np.array([Y_x, Y_y])
            else:
                if is_treat:
                    Y_x, Y_y = prev_state
                else:
                    dx, dy = 0.0, 0.0
                    for col in value_cols:
                        if pd.notna(row[col]) and pd.notna(prev_vals[col]):
                            delta = directions[col] * (row[col] - prev_vals[col]) * 0.1
                            if col in ['APTT', 'PT', 'TT', 'PLT', 'Hb', 'HCT', 'Fib']:
                                dx += delta
                            else:
                                dy += delta
                    
                    dx = np.clip(dx, -max_step_delta, max_step_delta)
                    dy = np.clip(dy, -max_step_delta, max_step_delta)
                    
                    # Cumulative restriction
                    if cumulative_dx + dx > total_delta_limit:
                        dx = total_delta_limit - cumulative_dx
                    elif cumulative_dx + dx < -total_delta_limit:
                        dx = -total_delta_limit - cumulative_dx
                    
                    if cumulative_dy + dy > total_delta_limit:
                        dy = total_delta_limit - cumulative_dy
                    elif cumulative_dy + dy < -total_delta_limit:
                        dy = -total_delta_limit - cumulative_dy
                    
                    Y_x = np.clip(prev_state[0] + dx, 0.0, 6.0)
                    Y_y = np.clip(prev_state[1] + dy, 0.0, 7.0)
                    cumulative_dx += dx
                    cumulative_dy += dy
                    prev_state = np.array([Y_x, Y_y])
            
            df.at[idx, "Y_x"] = Y_x
            df.at[idx, "Y_y"] = Y_y
            prev_vals = row[value_cols]
    
    return df